# Deep Learning Baseline for RUL Prediction

This notebook builds initial deep learning baselines for Remaining Useful Life prediction using time-windowed sensor data from C-MAPSS FD001. The objective is to compare sequence-based models (1D-CNN and GRU) against the best classical baseline from Step 4.

## 1. Imports and Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
DATA_DIR = '../data/processed'
METRICS_DIR = '../reports/metrics'
FIG_DIR = '../reports/figures'
PRED_DIR = '../reports/predictions'
MODEL_DIR = '../models/deep_learning'

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Load Processed Feature Set B Data

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train_b_fd001.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val_b_fd001.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test_b_fd001.csv')

print('Train:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)

In [ ]:
target_col = 'RUL_capped'
metadata_cols = ['unit_number', 'time_in_cycles', 'RUL', 'RUL_capped']

feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f'Number of input features: {len(feature_cols)}')
print(feature_cols)

In [ ]:
# Safety checks
for col in metadata_cols:
    assert col not in feature_cols, f'Metadata column found in features: {col}'
assert target_col in train_df.columns
print('All safety checks passed.')

## 3. Create Time-Windowed Sequence Dataset

Each training sample consists of sensor values over the previous N cycles. The target is the capped RUL at the final cycle of that window. A window size of 30 cycles is used as the initial setting. Units shorter than the window size are skipped.

In [ ]:
WINDOW_SIZE = 30

def create_windows(df, feature_cols, target_col, window_size=30, stride=1):
    X, y, meta = [], [], []

    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)

        feature_values = unit_df[feature_cols].values.astype(np.float32)
        target_values = unit_df[target_col].values.astype(np.float32)
        cycles = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values

        if len(unit_df) < window_size:
            continue

        for i in range(window_size, len(unit_df) + 1, stride):
            X.append(feature_values[i - window_size:i])
            y.append(target_values[i - 1])
            meta.append({
                'unit_number': unit,
                'time_in_cycles': cycles[i - 1],
                'RUL': raw_rul[i - 1],
                'RUL_capped': target_values[i - 1]
            })

    return np.array(X), np.array(y), pd.DataFrame(meta)

In [ ]:
X_train, y_train, meta_train = create_windows(train_df, feature_cols, target_col, WINDOW_SIZE)
X_val, y_val, meta_val = create_windows(val_df, feature_cols, target_col, WINDOW_SIZE)

print(f'X_train: {X_train.shape}  (samples, timesteps, features)')
print(f'y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'y_val:   {y_val.shape}')

## 4. Windowed Dataset Summary

In [ ]:
print(f'Training windows: {len(X_train)}')
print(f'Validation windows: {len(X_val)}')
print(f'Window size: {WINDOW_SIZE} cycles')
print(f'Features per timestep: {X_train.shape[2]}')
print(f'\nTarget (y_train) stats:')
print(f'  min={y_train.min():.1f}, max={y_train.max():.1f}, mean={y_train.mean():.1f}, std={y_train.std():.1f}')
print(f'\nUnits in train: {meta_train["unit_number"].nunique()}')
print(f'Units in val:   {meta_val["unit_number"].nunique()}')

## 5. Model Architecture: 1D-CNN

A simple 1D convolutional model that processes the time-windowed sensor inputs. Two convolutional layers extract local temporal patterns, followed by global pooling and dense layers for regression.

In [ ]:
def build_cnn_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

cnn_model = build_cnn_model((WINDOW_SIZE, len(feature_cols)))
cnn_model.summary()

## 6. Model Architecture: GRU

A GRU-based recurrent model that processes the sensor sequence. Two GRU layers capture temporal dependencies across the window, followed by dense layers for regression.

In [ ]:
def build_gru_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=True),
        layers.GRU(32),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

gru_model = build_gru_model((WINDOW_SIZE, len(feature_cols)))
gru_model.summary()

## 7. Training Configuration

In [ ]:
EPOCHS = 30
BATCH_SIZE = 256
PATIENCE = 10

def get_callbacks(model_name):
    return [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
    ]

## 8. Train 1D-CNN

In [ ]:
cnn_history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks('CNN1D'),
    verbose=1
)

## 9. Train GRU

In [ ]:
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks('GRU'),
    verbose=1
)

## 10. Training Curves

In [ ]:
def plot_training_curve(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].set_title(f'{model_name} - Loss')
    axes[0].legend()

    axes[1].plot(history.history['mae'], label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title(f'{model_name} - MAE')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/training_curve_{model_name}_fd001.png', dpi=150)
    plt.show()

plot_training_curve(cnn_history, 'CNN1D')
plot_training_curve(gru_history, 'GRU')

## 11. Evaluation

In [ ]:
def evaluate_model(model, X, y, model_name, feature_set='B', window_size=30):
    y_pred = model.predict(X, verbose=0).flatten()
    y_pred = np.clip(y_pred, 0, 125)

    rmse = root_mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    return {
        'model': model_name,
        'feature_set': feature_set,
        'window_size': window_size,
        'validation_rmse': round(rmse, 4),
        'validation_mae': round(mae, 4),
        'validation_r2': round(r2, 4),
    }, y_pred

cnn_results, cnn_preds = evaluate_model(cnn_model, X_val, y_val, 'CNN1D')
gru_results, gru_preds = evaluate_model(gru_model, X_val, y_val, 'GRU')

dl_results_df = pd.DataFrame([cnn_results, gru_results])
dl_results_df = dl_results_df.sort_values('validation_rmse').reset_index(drop=True)

dl_results_df.to_csv(f'{METRICS_DIR}/deep_learning_baseline_results_fd001.csv', index=False)
print(dl_results_df.to_string(index=False))

## 12. Save Models and Predictions

In [ ]:
cnn_model.save(f'{MODEL_DIR}/CNN1D_B_window30_fd001.keras')
gru_model.save(f'{MODEL_DIR}/GRU_B_window30_fd001.keras')
print('Models saved.')

# Save predictions
for model_name, preds in [('CNN1D', cnn_preds), ('GRU', gru_preds)]:
    pred_df = meta_val.copy()
    pred_df['prediction'] = preds
    pred_df['model'] = model_name
    pred_df.to_csv(f'{PRED_DIR}/val_predictions_{model_name}_B_window30_fd001.csv', index=False)

print('Predictions saved.')

## 13. Comparison with Classical Baseline

The best classical baseline from Step 4 was XGBoost on Feature Set C with validation RMSE 11.72 and MAE 8.32. This section compares the deep learning results against that reference.

In [ ]:
# Load classical results
classical_df = pd.read_csv(f'{METRICS_DIR}/classical_baseline_results_fd001.csv')
best_classical = classical_df.iloc[0]

comparison_rows = []
comparison_rows.append({
    'model': best_classical['model'],
    'feature_set': best_classical['feature_set'],
    'model_type': 'Classical',
    'validation_rmse': best_classical['validation_rmse'],
    'validation_mae': best_classical['validation_mae'],
    'validation_r2': best_classical['validation_r2'],
})

for _, row in dl_results_df.iterrows():
    comparison_rows.append({
        'model': row['model'],
        'feature_set': f"B (window={row['window_size']})",
        'model_type': 'Deep Learning',
        'validation_rmse': row['validation_rmse'],
        'validation_mae': row['validation_mae'],
        'validation_r2': row['validation_r2'],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('validation_rmse').reset_index(drop=True)
comparison_df.to_csv(f'{METRICS_DIR}/classical_vs_deep_baseline_comparison_fd001.csv', index=False)
print(comparison_df.to_string(index=False))

In [ ]:
# RMSE comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if t == 'Classical' else 'darkorange' for t in comparison_df['model_type']]
labels = comparison_df['model'] + ' / ' + comparison_df['feature_set']
ax.barh(labels, comparison_df['validation_rmse'], color=colors)
ax.set_xlabel('Validation RMSE')
ax.set_title('Classical vs Deep Learning Baseline Comparison - FD001')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/classical_vs_deep_rmse_comparison_fd001.png', dpi=150)
plt.show()

## 14. Actual vs Predicted and Error Distribution (Best Deep Model)

In [ ]:
# Identify best deep learning model
best_dl = dl_results_df.iloc[0]
best_dl_name = best_dl['model']
best_dl_preds = cnn_preds if best_dl_name == 'CNN1D' else gru_preds

print(f'Best deep learning model: {best_dl_name}')
print(f'  RMSE: {best_dl["validation_rmse"]}')
print(f'  MAE:  {best_dl["validation_mae"]}')
print(f'  R2:   {best_dl["validation_r2"]}')

In [ ]:
# Actual vs predicted
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_val, best_dl_preds, alpha=0.3, s=5)
ax.plot([0, 125], [0, 125], 'r--', linewidth=1)
ax.set_xlabel('Actual Capped RUL')
ax.set_ylabel('Predicted Capped RUL')
ax.set_title(f'Actual vs Predicted - {best_dl_name} (window={WINDOW_SIZE})')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/actual_vs_predicted_best_deep_fd001.png', dpi=150)
plt.show()

In [ ]:
# Error distribution
errors = best_dl_preds - y_val
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(errors, bins=40, edgecolor='black', alpha=0.7)
ax.set_xlabel('Prediction Error (predicted - actual)')
ax.set_ylabel('Frequency')
ax.set_title(f'Error Distribution - {best_dl_name} (window={WINDOW_SIZE})')
ax.axvline(x=0, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/error_distribution_best_deep_fd001.png', dpi=150)
plt.show()

print(f'Mean error: {errors.mean():.2f}')
print(f'Std error:  {errors.std():.2f}')

## 15. Sample Engine Prediction Trajectories

In [ ]:
sample_units = meta_val['unit_number'].unique()[:3]
best_pred_df = meta_val.copy()
best_pred_df['prediction'] = best_dl_preds

fig, ax = plt.subplots(figsize=(12, 6))
for unit in sample_units:
    unit_df = best_pred_df[best_pred_df['unit_number'] == unit].sort_values('time_in_cycles')
    ax.plot(unit_df['time_in_cycles'], unit_df['RUL_capped'],
            linewidth=1.5, label=f'Actual Unit {unit}')
    ax.plot(unit_df['time_in_cycles'], unit_df['prediction'],
            linestyle='--', linewidth=1.5, label=f'Predicted Unit {unit}')

ax.set_xlabel('Time in Cycles')
ax.set_ylabel('Capped RUL')
ax.set_title(f'Validation Engine Predictions - {best_dl_name} (window={WINDOW_SIZE})')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/sample_engine_prediction_trajectories_deep_fd001.png', dpi=150)
plt.show()

## 16. Window Size Sensitivity Check

A small sensitivity check to observe whether shorter or longer history windows affect validation performance. This is not full hyperparameter optimization.

In [ ]:
window_sizes = [15, 30, 50]
sensitivity_results = []

for ws in window_sizes:
    X_tr_w, y_tr_w, _ = create_windows(train_df, feature_cols, target_col, ws)
    X_va_w, y_va_w, _ = create_windows(val_df, feature_cols, target_col, ws)

    if len(X_tr_w) == 0 or len(X_va_w) == 0:
        print(f'Window size {ws}: skipped (insufficient data)')
        continue

    m = build_cnn_model((ws, len(feature_cols)))
    m.fit(
        X_tr_w, y_tr_w,
        validation_data=(X_va_w, y_va_w),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
        ],
        verbose=0
    )

    y_p = np.clip(m.predict(X_va_w, verbose=0).flatten(), 0, 125)
    rmse = root_mean_squared_error(y_va_w, y_p)
    mae = mean_absolute_error(y_va_w, y_p)
    r2 = r2_score(y_va_w, y_p)

    sensitivity_results.append({
        'window_size': ws,
        'validation_rmse': round(rmse, 4),
        'validation_mae': round(mae, 4),
        'validation_r2': round(r2, 4),
        'train_samples': len(X_tr_w),
        'val_samples': len(X_va_w),
    })
    print(f'Window size {ws}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}')

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_df.to_csv(f'{METRICS_DIR}/window_size_sensitivity_deep_fd001.csv', index=False)
print()
print(sensitivity_df.to_string(index=False))

## 17. Result Interpretation

The deep learning baselines use time-windowed selected sensor features (Feature Set B) as sequence inputs, while the classical baselines use hand-engineered degradation features (Feature Set C) as tabular inputs. The comparison is therefore not a direct like-for-like test, but rather a comparison of two different input representations for the same prediction task.

If the deep learning baseline did not outperform the strongest classical model, that is expected at this stage: the classical model benefits from explicitly engineered rolling statistics that encode degradation trends, while the sequence model must learn these patterns implicitly from raw sensor windows. The result still establishes a useful deep learning reference for the multi-view stage, where both input representations can be combined.

## 18. Observations

1. Time-windowed datasets were created using selected sensor features from Feature Set B.
2. Each input sample contains 30 cycles of sensor history and predicts the capped RUL at the final cycle of the window.
3. The train-validation split remains engine-level, consistent with the preprocessing strategy.
4. 1D-CNN and GRU baselines were trained to evaluate whether deep learning models can learn temporal degradation patterns from sensor history.
5. RMSE, MAE and R-squared were used for validation evaluation.
6. The best deep learning baseline is compared with the best classical baseline from Step 4.
7. The result will be used as a reference before developing the multi-view model.

In [ ]:
print('Generated artefacts:')
print()
print('Metrics:')
for f in sorted(os.listdir(METRICS_DIR)):
    if 'deep' in f or 'classical_vs' in f or 'window_size' in f:
        print(f'  {f}')
print()
print('Models:')
for f in sorted(os.listdir(MODEL_DIR)):
    print(f'  {f}')
print()
print('Predictions:')
for f in sorted(os.listdir(PRED_DIR)):
    if 'CNN1D' in f or 'GRU' in f:
        print(f'  {f}')
print()
print('Figures:')
for f in sorted(os.listdir(FIG_DIR)):
    if 'deep' in f or 'training_curve' in f or 'classical_vs_deep' in f:
        print(f'  {f}')